# Train NNUE with hyperparameter tuning (Kaggle)

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/chess-engine
!git clone https://github.com/bluepanda12323/chess-engine.git /kaggle/working/chess-engine
!pip install chess optuna zstandard
!ls /kaggle/working/chess-engine

In [ ]:
import io
import json
import math
import os
import time
import urllib.request

import numpy as np
import optuna
import torch
import zstandard
from matplotlib import pyplot as plt
from torch import nn

## Settings

In [ ]:
max_positions = 90_000_000
batch_size = 4096
num_trials = 20
tune_epochs = 3
tune_train_size = 10_000_000
tune_val_size = 1_000_000
hidden_options = [384, 512, 768, 1024]
layer2_options = [0, 8, 16, 32]
max_cost_factor = 4
num_epochs = 20

url = "https://database.lichess.org/lichess_db_eval.jsonl.zst"
out_dir = "/kaggle/working"
log_file = os.path.join(out_dir, "nnue_tuned.log")
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

print("device:", device)

## Helpers

In [ ]:
def log(text):
    line = f"[{time.strftime('%H:%M:%S')}] {text}"
    print(line, flush=True)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")


def compute_cost(num_hidden, num_layer2):
    update = 8 * num_hidden
    if num_layer2 == 0:
        return update + 2 * num_hidden
    return update + 2 * num_hidden * num_layer2 + num_layer2


current_cost = compute_cost(256, 0)
max_cost = max_cost_factor * current_cost
architectures = [f"{h}x{l}" for h in hidden_options for l in layer2_options if compute_cost(h, l) <= max_cost]

print("current model (256x0) cost:", current_cost, "| max cost:", max_cost)
print("allowed architectures:", architectures)

## Loading the data: streamed from Lichess, quiet positions only

In [ ]:
PIECE_CODES = {"P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6, "p": 7, "n": 8, "b": 9, "r": 10, "q": 11, "k": 12}
DIGITS = "12345678"


def parse_fen(fen):
    board_part, color = fen.split(" ")[:2]
    row = np.zeros(64, dtype=np.uint8)
    square = 0
    for char in board_part:
        if char == "/":
            continue
        if char in DIGITS:
            square += int(char)
        else:
            row[square] = PIECE_CODES[char]
            square += 1
    return row, color == "w"


def square_index(square):
    return (8 - int(square[1])) * 8 + (ord(square[0]) - ord("a"))


def is_quiet(row, move):
    if len(move) == 5:
        return False
    from_square, to_square = square_index(move[:2]), square_index(move[2:4])
    if row[to_square] != 0:
        return False
    if row[from_square] in (1, 7) and move[0] != move[2]:
        return False
    return True


def score_to_label(cp, mate):
    if mate is not None:
        return 1.0 if mate > 0 else -1.0
    return 2 / (1 + math.exp(-0.00368208 * cp)) - 1


def load_positions(max_positions):
    codes = np.zeros((max_positions, 64), dtype=np.uint8)
    side = np.zeros(max_positions, dtype=np.uint8)
    labels = np.zeros(max_positions, dtype=np.float32)

    idx, skipped = 0, 0
    with urllib.request.urlopen(url) as response:
        reader = io.TextIOWrapper(zstandard.ZstdDecompressor().stream_reader(response), encoding="utf-8")
        for line in reader:
            if idx >= max_positions:
                break
            data = json.loads(line)
            if not data["evals"]:
                continue
            deepest = max(data["evals"], key=lambda e: e["depth"])
            pv = deepest["pvs"][0]
            if not pv.get("line"):
                continue
            row, white = parse_fen(data["fen"])
            if not is_quiet(row, pv["line"].split()[0]):
                skipped += 1
                continue
            codes[idx], side[idx] = row, white
            labels[idx] = score_to_label(pv.get("cp"), pv.get("mate"))
            idx += 1
            if idx % 1_000_000 == 0:
                log(f"Loaded {idx:,} / {max_positions:,} quiet positions ({skipped:,} skipped)")

    log(f"Done: {idx:,} quiet positions, {skipped:,} skipped because the best move was a capture or promotion")
    return codes[:idx], side[:idx], labels[:idx]

## Features: the board from both perspectives

In [ ]:
NUM_FEATURES = 768

WHITE_FEATURES = torch.full((13, 64), NUM_FEATURES, dtype=torch.long)
BLACK_FEATURES = torch.full((13, 64), NUM_FEATURES, dtype=torch.long)
for code in range(1, 13):
    piece_type = (code - 1) % 6
    is_white = code <= 6
    for index in range(64):
        square = index ^ 56
        WHITE_FEATURES[code, index] = (0 if is_white else 384) + piece_type * 64 + square
        BLACK_FEATURES[code, index] = (384 if is_white else 0) + piece_type * 64 + (square ^ 56)

WHITE_FEATURES = WHITE_FEATURES.to(device)
BLACK_FEATURES = BLACK_FEATURES.to(device)
SQUARES = torch.arange(64, device=device)


def one_hot(table, data_codes):
    indices = table[data_codes.long(), SQUARES]
    x = torch.zeros((data_codes.shape[0], NUM_FEATURES + 1), device=device)
    x.scatter_(1, indices, 1.0)
    return x[:, :NUM_FEATURES]


def features(data_codes, data_side):
    x_white = one_hot(WHITE_FEATURES, data_codes)
    x_black = one_hot(BLACK_FEATURES, data_codes)
    white_to_move = data_side.bool().unsqueeze(1)
    x_us = torch.where(white_to_move, x_white, x_black)
    x_them = torch.where(white_to_move, x_black, x_white)
    return x_us, x_them

## Model

In [ ]:
class NNUE(nn.Module):

    def __init__(self, num_features, num_hidden, num_layer2):
        super().__init__()
        self.feature_transformer = nn.Linear(in_features=num_features, out_features=num_hidden)
        if num_layer2 > 0:
            self.hidden = nn.Linear(in_features=2 * num_hidden, out_features=num_layer2)
            self.output = nn.Linear(in_features=num_layer2, out_features=1)
        else:
            self.hidden = None
            self.output = nn.Linear(in_features=2 * num_hidden, out_features=1)
        self.output_act_fn = nn.Tanh()

    def forward(self, x_us, x_them):
        us = torch.clamp(self.feature_transformer(x_us), 0, 1)
        them = torch.clamp(self.feature_transformer(x_them), 0, 1)
        x = torch.cat([us, them], dim=1)
        if self.hidden is not None:
            x = torch.clamp(self.hidden(x), 0, 1)
        x = self.output(x)
        x = self.output_act_fn(x)
        return x[:, 0]


def build_model(architecture):
    num_hidden, num_layer2 = (int(part) for part in architecture.split("x"))
    return NNUE(num_features=NUM_FEATURES, num_hidden=num_hidden, num_layer2=num_layer2).to(device)

## Data

In [ ]:
log(f"Start on {device}, streaming data from Lichess...")
codes, side, labels = load_positions(max_positions)

num_positions = len(labels)
perm = torch.randperm(num_positions, generator=torch.Generator().manual_seed(42)).numpy()
num_train = int(0.8 * num_positions)
num_val = int(0.1 * num_positions)
train_idx = perm[:num_train]
val_idx = perm[num_train:num_train + num_val]
test_idx = perm[num_train + num_val:]
tune_train_idx = train_idx[:tune_train_size]
tune_val_idx = val_idx[:tune_val_size]
log(f"Train: {len(train_idx):,}, val: {len(val_idx):,}, test: {len(test_idx):,}")


def batches(indices, shuffle):
    if shuffle:
        indices = indices[torch.randperm(len(indices)).numpy()]
    for start in range(0, len(indices), batch_size):
        batch = np.sort(indices[start:start + batch_size])
        data_codes = torch.from_numpy(codes[batch]).to(device)
        data_side = torch.from_numpy(side[batch]).to(device)
        data_labels = torch.from_numpy(labels[batch]).to(device)
        x_us, x_them = features(data_codes, data_side)
        data_labels = torch.where(data_side.bool(), data_labels, -data_labels)
        yield x_us, x_them, data_labels

## Training functions

In [ ]:
loss_fn = nn.MSELoss()


def train_one_epoch(model, optimizer, indices):
    model.train()
    total_loss, num_batches = 0., 0

    for x_us, x_them, data_labels in batches(indices, shuffle=True):
        preds = model(x_us, x_them)
        loss = loss_fn(preds, data_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

    return total_loss / num_batches


def eval_model(model, indices):
    model.eval()
    total_loss, num_batches = 0., 0
    total_abs_error, num_preds = 0., 0

    with torch.no_grad():
        for x_us, x_them, data_labels in batches(indices, shuffle=False):
            preds = model(x_us, x_them)

            total_loss += loss_fn(preds, data_labels).item()
            num_batches += 1

            total_abs_error += (preds - data_labels).abs().sum().item()
            num_preds += data_labels.shape[0]

    return total_loss / num_batches, total_abs_error / num_preds

## Hyperparameter tuning

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
    architecture = trial.suggest_categorical("architecture", architectures)
    learning_rate = trial.suggest_float("learning_rate", 3e-4, 3e-3, log=True)

    model = build_model(architecture)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(tune_epochs):
        train_one_epoch(model, optimizer, tune_train_idx)
        val_loss, val_mae = eval_model(model, tune_val_idx)
        trial.report(val_loss, epoch)
        if trial.should_prune():
            log(f"Trial {trial.number}: stopped early after epoch {epoch + 1} (val loss {val_loss:.4f}, {architecture}, lr {learning_rate:.5f})")
            raise optuna.TrialPruned()

    log(f"Trial {trial.number}: val loss {val_loss:.4f}, val MAE {val_mae:.4f}, {architecture}, lr {learning_rate:.5f}")
    return val_loss


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
)
study.optimize(objective, n_trials=num_trials)

best_params = study.best_params
log(f"Best params: {best_params} (val loss {study.best_value:.4f})")
print(study.trials_dataframe(attrs=("number", "value", "params", "state")).sort_values("value"))

## Final training

In [ ]:
num_hidden, num_layer2 = (int(part) for part in best_params["architecture"].split("x"))
model = build_model(best_params["architecture"])
optimizer = torch.optim.Adam(model.parameters(), lr=best_params["learning_rate"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=best_params["learning_rate"] / 100)
model_path = os.path.join(out_dir, "nnue_weights.npz")
print(model)


def save_weights(model, path):
    weights = {name: value.detach().cpu().numpy() for name, value in model.state_dict().items()}
    np.savez(path, num_hidden=np.array(num_hidden), num_layer2=np.array(num_layer2), **weights)


history = {"train_loss": [], "val_loss": [], "val_mae": [], "learning_rate": []}
best_val_loss = float("inf")
best_state = None

for epoch in range(num_epochs):
    current_lr = optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(model, optimizer, train_idx)
    val_loss, val_mae = eval_model(model, val_idx)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_mae"].append(val_mae)
    history["learning_rate"].append(current_lr)
    log(f"Epoch {epoch + 1}: train loss {train_loss:.4f}, val loss {val_loss:.4f}, val MAE {val_mae:.4f}, lr {current_lr:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {name: value.detach().clone() for name, value in model.state_dict().items()}
        save_weights(model, model_path)
        log("Checkpoint saved")

## Test

In [ ]:
model.load_state_dict(best_state)
test_loss, test_mae = eval_model(model, test_idx)
log(f"Test: loss {test_loss:.4f}, MAE {test_mae:.4f}")

params = {
    "architecture": best_params["architecture"],
    "num_hidden": num_hidden,
    "num_layer2": num_layer2,
    "learning_rate": best_params["learning_rate"],
    "cost": compute_cost(num_hidden, num_layer2),
    "max_cost": max_cost,
    "num_positions": num_positions,
}
with open(os.path.join(out_dir, "nnue_tuned.json"), "w", encoding="utf-8") as f:
    json.dump({"params": params, "history": history, "test_loss": test_loss, "test_mae": test_mae}, f, indent=2)
log("Done")

## Sanity check

In [ ]:
test_fens = {
    "start position": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "white a bishop down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQK1NR w KQkq - 0 1",
    "white a queen down, white to move": "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNB1KBNR w KQkq - 0 1",
    "lost game, black a bishop down (Stockfish -639)": "2r3k1/5p1p/bq2p1p1/pN1pP3/P2P2n1/RP1Q1NP1/5PBP/6K1 b - - 0 22",
}

model.eval()
for name, fen in test_fens.items():
    row, white = parse_fen(fen)
    data_codes = torch.from_numpy(row).unsqueeze(0).to(device)
    data_side = torch.tensor([white], dtype=torch.uint8, device=device)
    with torch.no_grad():
        value = model(*features(data_codes, data_side)).item()
    cp = math.atanh(max(min(value, 0.9999), -0.9999)) / 0.00184104
    log(f"{name}: {value:+.3f} for the side to move (about {cp:+.0f} cp)")

## Plots

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(epochs, history["val_mae"], label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(epochs, history["learning_rate"], label="Learning rate")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.yscale("log")
plt.legend()

plt.savefig(os.path.join(out_dir, "nnue_tuned_curves.png"), bbox_inches="tight")
plt.show()

In [ ]:
try:
    optuna.visualization.matplotlib.plot_optimization_history(study)
    plt.savefig(os.path.join(out_dir, "tuning_history.png"), bbox_inches="tight")
    plt.show()
    optuna.visualization.matplotlib.plot_param_importances(study)
    plt.savefig(os.path.join(out_dir, "tuning_importances.png"), bbox_inches="tight")
    plt.show()
except Exception as error:
    log(f"Could not make tuning plots: {error}")